# LAB | Imbalanced

**Load the data**

In this challenge, we will be working with Credit Card Fraud dataset.

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv

Metadata

- **distance_from_home:** the distance from home where the transaction happened.
- **distance_from_last_transaction:** the distance from last transaction happened.
- **ratio_to_median_purchase_price:** Ratio of purchased price transaction to median purchase price.
- **repeat_retailer:** Is the transaction happened from same retailer.
- **used_chip:** Is the transaction through chip (credit card).
- **used_pin_number:** Is the transaction happened by using PIN number.
- **online_order:** Is the transaction an online order.
- **fraud:** Is the transaction fraudulent. **0=legit** -  **1=fraud**


In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [4]:
fraud = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv")
fraud.head()

,distance_from_home,distance_from_last_transaction,ratio_to_median_purchase_price,repeat_retailer,used_chip,used_pin_number,online_order,fraud
0,57.877857,0.311140,1.945940,1.0,1.0,0.0,0.0,0.0
1,10.829943,0.175592,1.294219,1.0,0.0,0.0,0.0,0.0
2,5.091079,0.805153,0.427715,1.0,0.0,0.0,1.0,0.0
3,2.247564,5.600044,0.362663,1.0,1.0,0.0,1.0,0.0
4,44.190936,0.566486,2.222767,1.0,1.0,0.0,1.0,0.0


**Steps:**

- **1.** What is the distribution of our target variable? Can we say we're dealing with an imbalanced dataset?
- **2.** Train a LogisticRegression.
- **3.** Evaluate your model. Take in consideration class importance, and evaluate it by selection the correct metric.
- **4.** Run **Oversample** in order to balance our target variable and repeat the steps above, now with balanced data. Does it improve the performance of our model? 
- **5.** Now, run **Undersample** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model?
- **6.** Finally, run **SMOTE** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model? 

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, f1_score

# Install imblearn via: pip install imbalanced-learn
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler

# 1. Load data and check distribution
url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv"
df = pd.read_csv(url)

print("--- Class Distribution ---")
print(df['fraud'].value_counts())
print(df['fraud'].value_counts(normalize=True) * 100)

# Split features and target
X = df.drop(columns=['fraud'])
y = df['fraud']

# Train-test split (stratified to retain distribution)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Feature Scaling (Crucial for Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

def evaluate_pipeline(X_tr, y_tr, name):
    # Train Logistic Regression
    model = LogisticRegression(max_iter=1000, random_state=42)
    model.fit(X_tr, y_tr)
    
    # Predict and evaluate
    preds = model.predict(X_test_scaled)
    probs = model.predict_proba(X_test_scaled)[:, 1]
    
    print(f"\n==================== {name} ====================")
    print(classification_report(y_test, preds, digits=4))
    print(f"ROC AUC Score: {roc_auc_score(y_test, probs):.4f}")
    print(f"Minority F1-Score: {f1_score(y_test, preds):.4f}")

# 2 & 3. Baseline Logistic Regression
evaluate_pipeline(X_train_scaled, y_train, "Baseline Logistic Regression")

# 4. Oversampling
ros = RandomOverSampler(random_state=42)
X_train_over, y_train_over = ros.fit_resample(X_train_scaled, y_train)
evaluate_pipeline(X_train_over, y_train_over, "Oversampled Logistic Regression")

# 5. Undersampling
rus = RandomUnderSampler(random_state=42)
X_train_under, y_train_under = rus.fit_resample(X_train_scaled, y_train)
evaluate_pipeline(X_train_under, y_train_under, "Undersampled Logistic Regression")

# 6. SMOTE
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)
evaluate_pipeline(X_train_smote, y_train_smote, "SMOTE Balanced Logistic Regression")

--- Class Distribution ---
fraud
0.0    912597
1.0     87403
Name: count, dtype: int64
fraud
0.0    91.2597
1.0     8.7403
Name: proportion, dtype: float64

==================== Baseline Logistic Regression ====================
              precision    recall  f1-score   support

         0.0     0.9632    0.9933    0.9780    273779
         1.0     0.8959    0.6033    0.7210     26221

    accuracy                         0.9592    300000
   macro avg     0.9295    0.7983    0.8495    300000
weighted avg     0.9573    0.9592    0.9555    300000

ROC AUC Score: 0.9671
Minority F1-Score: 0.7210

==================== Oversampled Logistic Regression ====================
              precision    recall  f1-score   support

         0.0     0.9947    0.9333    0.9630    273779
         1.0     0.5765    0.9482    0.7171     26221

    accuracy                         0.9346    300000
   macro avg     0.7856    0.9408    0.8400    300000
weighted avg     0.9582    0.9346    0.9415    300


Precision: how often is the model actually right? 
Recall: Out of the actual fraud happening, how much did the model catch? 

ROC AUC stand for Receiver Operating Characteristic -area under the curve
it measures how well the model can distinguish between the tow classes (fraud vs normal) across al possible confidence levels 
our baseline model scored 0.8205. This means if you pick one random actual fraud transaction and one random normal transaction, there is an 82% chance the model will correctly rank the fraud transactions as "more suspicious" than the normal one. 
F1-Score is the harmonic mean of Precision and Recall. It punished models if either of the two are terrible.
Minority F1-score - how well it handles the rare class